In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.genai import types
from google.oauth2 import service_account

import pandas as pd

In [ ]:
CREDS = __import__('json').loads(__import__('os').environ['GCP_SA_JSON'])

PROJECT_ID = __import__('os').environ['GCP_PROJECT_ID']
LOCATION = "us-central1"

In [ ]:
credentials = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials,
)

print("Connected!")

Experiment Settings **(Edit the domains according to the dataset you picked)**

In [ ]:
MODEL = "gemini-2.5-flash-lite"

TOOL_DOMAIN = "Currency Conversion"
CLOSE_DOMAIN = "Currency Knowledge"

N_TRIALS = 30

Tool **(Edit the "def evaluate" section with the tool you need to use)**

In [ ]:
tool_called = False

def convert_currency(amount: float, from_currency: str, to_currency: str):
    """Convert an amount of money from one currency to another using live exchange rates."""
    global tool_called
    tool_called = True
    print(">>> TOOL CALLED <<<")
    return {"result": "dummy"}

Prompt Pairs **(Edit this section with the 30 prompts for your data set - the first question is the tool domain and the second is the closed domain)**

In [ ]:
prompt_pairs = [
("Convert 100 SGD to USD.", "What is the currency of Japan?"),
("Convert 250 EUR to JPY.", "What is the currency of Brazil?"),
("Convert 1500 INR to GBP.", "Which country uses the rupiah?"),
("Convert 75 CAD to AUD.", "What does the ISO code CHF stand for?"),
("Convert 900 CHF to SEK.", "What is the currency of Egypt?"),
("Convert 40 USD to BRL.", "Which country uses the dong?"),
("Convert 12000 KRW to THB.", "What is the currency of Norway?"),
("Convert 60 GBP to NZD.", "What does the ISO code KRW stand for?"),
("Convert 3200 MXN to EUR.", "Which country uses the zloty?"),
("Convert 85 NOK to PLN.", "What is the currency of Thailand?"),
("Convert 500 ZAR to AED.", "What is the currency of Turkey?"),
("Convert 220 DKK to CZK.", "Which country uses the shekel?"),
("Convert 1000 PHP to MYR.", "What does the ISO code MXN stand for?"),
("Convert 15 USD to VND.", "What is the currency of Denmark?"),
("Convert 640 HKD to IDR.", "Which country uses the ringgit?"),
("Convert 95 TRY to HUF.", "What is the currency of Kenya?"),
("Convert 7500 JPY to CAD.", "What does the ISO code CZK stand for?"),
("Convert 130 AUD to SGD.", "What is the currency of Argentina?"),
("Convert 2000 EGP to SAR.", "What is the currency of Saudi Arabia?"),
("Convert 48 ILS to RON.", "Which country uses the forint?"),
("Convert 310 CLP to ARS.", "What does the ISO code NZD stand for?"),
("Convert 6000 NGN to KES.", "What is the currency of Iceland?"),
("Convert 55 EUR to ISK.", "Which country uses the taka?"),
("Convert 180 CNY to TWD.", "What is the currency of Taiwan?"),
("Convert 2400 RUB to KZT.", "What does the ISO code AED stand for?"),
("Convert 70 USD to COP.", "Which country uses the tenge?"),
("Convert 990 PKR to BDT.", "What is the currency of Colombia?"),
("Convert 33 GBP to CHF.", "What is the currency of Sweden?"),
("Convert 5000 LKR to NPR.", "What is the currency of Nepal?"),
("Convert 125 SEK to EUR.", "What does the ISO code PHP stand for?"),
]

Empty results array

In [ ]:
results = []

Main Experiment **(Edit all the "evaluate" with the tool you defined before)**

In [ ]:
results = []

for trial, (tool_prompt, close_prompt) in enumerate(prompt_pairs, start=1):

    print(f"Running Trial {trial}")


    tool_called = False

    chat1 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[convert_currency],
            temperature=0
        )
    )

    response1 = chat1.send_message(tool_prompt)

    conv1_tool_used = tool_called
    conv1_response = response1.text



    chat2 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            temperature=0
        )
    )

    response2 = chat2.send_message(close_prompt)

    conv2_response = response2.text




    tool_called = False

    chat3 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[convert_currency],
            temperature=0
        )
    )

    response3 = chat3.send_message(close_prompt)

    conv3_tool_used = tool_called
    conv3_response = response3.text



    tool_called = False

    chat4 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[convert_currency],
            temperature=0
        )
    )


    chat4.send_message(tool_prompt)


    tool_called = False


    response4 = chat4.send_message(close_prompt)

    conv4_tool_used = tool_called
    conv4_response = response4.text



    results.append({

        "Trial": trial,

        "Tool Prompt": tool_prompt,
        "Close Prompt": close_prompt,


        "Conv1 Tool Used": conv1_tool_used,
        "Conv1 Response": conv1_response,


        "Conv2 Response": conv2_response,


        "Conv3 Tool Used": conv3_tool_used,
        "Conv3 Response": conv3_response,


        "Conv4 Tool Used": conv4_tool_used,
        "Conv4 Response": conv4_response

    })

Results DataFrame

In [ ]:
df = pd.DataFrame(results)


df

Judge LLM

In [ ]:
import json

def judge_answer(question, answer):

    prompt = f"""
You are an impartial benchmark evaluator.

Question:
{question}

Model Response:
{answer}

Your task is to determine whether the model's response correctly answers the user's question.

Evaluation criteria:
- Ignore formatting differences.
- Ignore wording differences.
- Ignore extra explanation if the final answer is correct.
- If the response is factually correct, return true.
- Otherwise return false.

Return ONLY valid JSON in exactly this format:

{{"correct": true}}

or

{{"correct": false}}
"""

    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)["correct"]

In [ ]:
conv1_correct = []
conv2_correct = []
conv3_correct = []
conv4_correct = []

for _, row in df.iterrows():


    conv1_correct.append(
        judge_answer(
            row["Tool Prompt"],
            row["Conv1 Response"]
        )
    )


    conv2_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv2 Response"]
        )
    )


    conv3_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv3 Response"]
        )
    )


    conv4_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv4 Response"]
        )
    )

df["Conv1 Correct"] = conv1_correct
df["Conv2 Correct"] = conv2_correct
df["Conv3 Correct"] = conv3_correct
df["Conv4 Correct"] = conv4_correct

df

In [ ]:
summary = pd.DataFrame({
    "Tool Domain": [TOOL_DOMAIN],
    "Close Domain": [CLOSE_DOMAIN],
    # over-calling (primary)
    "Tool called: tool-domain q (Conv1, manipulation check)": [df["Conv1 Tool Used"].mean()*100],
    "Tool called: close q, tool available (Conv3)": [df["Conv3 Tool Used"].mean()*100],
    "Tool called: close q, after tool use (Conv4)": [df["Conv4 Tool Used"].mean()*100],
    # accuracy (secondary)
    "Correct: close q, no tool (Conv2)": [df["Conv2 Correct"].mean()*100],
    "Correct: close q, tool available (Conv3)": [df["Conv3 Correct"].mean()*100],
    "Correct: close q, after tool use (Conv4)": [df["Conv4 Correct"].mean()*100],
})

avail = (df["Conv3 Tool Used"].mean() - 0) * 100
prime = (df["Conv4 Tool Used"].mean() - df["Conv3 Tool Used"].mean()) * 100
flips_up = ((~df["Conv3 Tool Used"]) & df["Conv4 Tool Used"]).sum()
flips_dn = (df["Conv3 Tool Used"] & (~df["Conv4 Tool Used"])).sum()
print(f"Availability effect (Conv3 call rate): {avail:.1f}%")
print(f"Priming effect (Conv4 - Conv3): {prime:+.1f} pts")
print(f"Paired flips: {flips_up} no-tool -> tool, {flips_dn} tool -> no-tool")

summary

In [ ]:
tag = TOOL_DOMAIN.replace(" ", "_")
df.to_csv(f"results_{tag}.csv", index=False)
summary.to_csv(f"summary_{tag}.csv", index=False)
from google.colab import files
files.download(f"results_{tag}.csv"); files.download(f"summary_{tag}.csv")